# PJM API — local debug notebook

Interactive smoke test for **pjm-api**. Fill in credentials in the next cell, then run all cells.

**Prerequisites**

```bash
python -m pip install -e ".[pfx]"
```

Use your **login** `.p12` / `.pfx` file (not the public `.cer` upload). Passwords and tokens are redacted in output.

In [ ]:
from __future__ import annotations

import json
import logging
import sys
import traceback
from pathlib import Path

from pjm_api.certs import inspect_certificate
from pjm_api.config import load_settings, parse_certificate
from pjm_api.doctor import format_doctor_report, run_doctor
from pjm_api.logging_utils import configure_logging, redact_secrets
from pjm_api.oasis import OasisClient, build_template_url

configure_logging(verbose=True)
logging.getLogger("pjm_api").setLevel(logging.DEBUG)


def mask(value: str, *, show: int = 4) -> str:
    if not value:
        return "(empty)"
    if len(value) <= show * 2:
        return "*" * len(value)
    return f"{value[:show]}...{value[-show:]} ({len(value)} chars)"


def section(title: str) -> None:
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def dump_settings(settings) -> None:
  section("Resolved settings")
  rows = {
      "username": settings.username,
      "password": mask(settings.password),
      "environment": settings.environment,
      "certificate_path": str(settings.certificate_path) if settings.certificate_path else None,
      "certificate_password": mask(settings.certificate_password or ""),
      "oasis_base_url": settings.oasis_base_url,
      "sso_url": settings.sso_url,
      "logout_url": settings.logout_url,
      "backend": settings.backend,
      "timeout_sec": settings.timeout_sec,
      "downloads_dir": str(settings.downloads_dir),
  }
  for key, value in rows.items():
      print(f"  {key:22} {value}")
  missing = settings.missing_for_backend()
  if missing:
      print(f"\n  MISSING: {', '.join(missing)}")
  else:
      print("\n  All required fields present.")


def dump_cert_report(report) -> None:
    section("Certificate inspection")
    print(f"  path              {report.path}")
    print(f"  exists            {report.path.exists()}")
    print(f"  kind              {report.kind.value}")
    print(f"  healthy           {report.healthy}")
    print(f"  subject           {report.subject or '(n/a)'}")
    print(f"  issuer            {report.issuer or '(n/a)'}")
    print(f"  thumbprint        {report.thumbprint or '(n/a)'}")
    print(f"  not_before        {report.not_before}")
    print(f"  not_after         {report.not_after}")
    print(f"  days_until_expiry {report.days_until_expiry}")
    if report.warnings:
        print("  warnings:")
        for w in report.warnings:
            print(f"    - {w}")
    if report.errors:
        print("  errors:")
        for e in report.errors:
            print(f"    - {e}")


def dump_response(response, *, preview_chars: int = 2000) -> None:
    section("OASIS response")
    print(f"  status_code    {response.status_code}")
    print(f"  ok             {response.ok}")
    print(f"  template       {response.template}")
    print(f"  environment    {response.environment}")
    print(f"  output_format  {response.output_format}")
    print(f"  content_bytes  {len(response.content)}")
    print("  headers:")
    for name, value in sorted(response.headers.items()):
        print(f"    {name}: {redact_secrets(value)}")
    text = response.text()
    preview = text[:preview_chars]
    print(f"\n  body preview ({min(len(text), preview_chars)} of {len(text)} chars):")
    print(redact_secrets(preview))
    if len(text) > preview_chars:
        print("  ... (truncated)")


def run_step(title: str, fn) -> bool:
    section(title)
    try:
        fn()
        print("  => OK")
        return True
    except Exception as exc:
        print(f"  => FAIL: {type(exc).__name__}: {exc}")
        if getattr(exc, "fix", None):
            print(f"  Fix: {exc.fix}")
        traceback.print_exc()
        return False

In [ ]:
# --- Edit these values, then run all cells below ---

USERNAME = ""
PASSWORD = ""

# Path to login .p12 / .pfx (or "path|password" in one string)
CERTIFICATE_PATH = ""

# Optional if password is not embedded in CERTIFICATE_PATH
CERTIFICATE_PASSWORD = ""

# TRAIN | STAGE | TEST | PRODUCTION
ENVIRONMENT = "TRAIN"

# ---

if not USERNAME or not PASSWORD or not CERTIFICATE_PATH:
    raise ValueError("Set USERNAME, PASSWORD, and CERTIFICATE_PATH before continuing.")

cert_path, embedded_password = parse_certificate(CERTIFICATE_PATH)
cert_password = CERTIFICATE_PASSWORD or embedded_password or ""

settings = load_settings(
    username=USERNAME,
    password=PASSWORD,
    certificate=str(cert_path),
    certificate_password=cert_password,
    environment=ENVIRONMENT,
    use_credentials_file=False,
    prompt_unlock=False,
)

dump_settings(settings)

In [ ]:
report = inspect_certificate(settings.certificate_path, settings.certificate_password)
dump_cert_report(report)

if not report.healthy:
    raise RuntimeError("Certificate inspection failed — fix certificate before continuing.")

In [ ]:
offline_steps, offline_ok = run_doctor(settings, offline=True)
section("Doctor (offline)")
print(format_doctor_report(offline_steps, offline_ok, offline=True))

if not offline_ok:
    raise RuntimeError("Offline doctor checks failed.")

In [ ]:
def _sso_debug() -> None:
    with OasisClient(settings) as client:
        token = client.session.token
        print(f"  SSO URL          {settings.sso_url}")
        print(f"  authenticated    {client.session.is_authenticated}")
        print(f"  token            {mask(token or '')}")
        cookie = client.session.cookie_header
        print(f"  cookie header    {redact_secrets(cookie)}")


sso_ok = run_step("SSO authentication", _sso_debug)
if not sso_ok:
    raise RuntimeError("SSO authentication failed.")

In [ ]:
TRANSSERV_PARAMS = {
    "OUTPUT_FORMAT": "DATA",
    "PRIMARY_PROVIDER_CODE": "PJM",
    "PRIMARY_PROVIDER_DUNS": "073647877",
    "RETURN_TZ": "EP",
    "VERSION": "3.3",
}

request_url = build_template_url(settings.oasis_base_url, "TRANSSERV")
section("TRANSSERV smoke request")
print(f"  base URL         {settings.oasis_base_url}")
print(f"  template URL     {request_url}")
print(f"  params           {json.dumps(TRANSSERV_PARAMS, indent=2)}")


def _transserv_smoke() -> None:
    with OasisClient(settings) as client:
        response = client.request("TRANSSERV", TRANSSERV_PARAMS)
        dump_response(response)


smoke_ok = run_step("TRANSSERV request", _transserv_smoke)
if not smoke_ok:
    raise RuntimeError("TRANSSERV smoke request failed.")

In [ ]:
online_steps, online_ok = run_doctor(settings, offline=False)
section("Doctor (full — SSO + TRANSSERV)")
print(format_doctor_report(online_steps, online_ok))

section("Summary")
print(f"  Python           {sys.version.split()[0]}")
print(f"  pjm_api          {__import__('pjm_api').__version__}")
print(f"  environment      {settings.environment}")
print(f"  offline doctor   {'PASS' if offline_ok else 'FAIL'}")
print(f"  SSO              {'PASS' if sso_ok else 'FAIL'}")
print(f"  TRANSSERV smoke  {'PASS' if smoke_ok else 'FAIL'}")
print(f"  full doctor      {'PASS' if online_ok else 'FAIL'}")

if online_ok:
    print("\nAll checks passed. Credentials and certificate look good.")
else:
    print("\nSomething failed — see sections above and docs/troubleshooting.md")